# Notebook 04 — Condition C: LSTM with Simple Imputation

## Requires
- `data/raw/`

## Produces
- `results/models/lstm_condition_C.pt`
- Row appended to `results/experiment_log.csv`

> **GPU required:** Runtime > Change runtime type > T4 GPU

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import torch

from src.config import RANDOM_SEED, DATA_DIR, MODELS_DIR
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers, apply_strategy_A
from src.utils import set_all_seeds, create_patient_splits
from src.models import SepsisLSTM
from src.train import make_loaders, train_lstm
from src.evaluate import select_threshold, compute_all_metrics, log_results

set_all_seeds(RANDOM_SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print('Imports OK')

## 1. Load Data and Create Splits

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df['patient_id'].isin(patient_train)].copy()
val_df   = df[df['patient_id'].isin(patient_val)].copy()
test_df  = df[df['patient_id'].isin(patient_test)].copy()

print(f'Train: {len(patient_train):,} | Val: {len(patient_val):,} | Test: {len(patient_test):,}')

## 2. Strategy A Preprocessing

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = apply_strategy_A(
    train_df, val_df, test_df
)

# Rebuild DataFrames for SepsisDataset (needs patient_id, ICULOS, EarlyLabel + features).
# ICULOS is in feature_cols (part of ALL_FEATURES) so only patient_id + EarlyLabel as meta.
def make_scaled_df(original_df, X_scaled, feat_cols):
    meta = original_df[['patient_id', 'EarlyLabel']].reset_index(drop=True)
    return pd.concat([meta, pd.DataFrame(X_scaled, columns=feat_cols)], axis=1)

train_lstm_df = make_scaled_df(train_df, X_train, feature_cols)
val_lstm_df   = make_scaled_df(val_df,   X_val,   feature_cols)
test_lstm_df  = make_scaled_df(test_df,  X_test,  feature_cols)

assert 'ICULOS' in train_lstm_df.columns, 'ICULOS missing from feature_cols'
print(f'Device: {device}  Features: {len(feature_cols)}')

## 3. Hyperparameter Grid Search

In [ ]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f'pos_weight = {pos_weight:.1f}')

# Build loaders ONCE — reused across all combinations
train_loader, val_loader, test_loader = make_loaders(
    train_lstm_df, val_lstm_df, test_lstm_df,
    patient_train, patient_val, patient_test,
    feature_cols=feature_cols, batch_size=64,
)
print('Loaders ready')

param_grid = [{'hidden_size': h, 'dropout': d, 'lr': lr}
              for h in [64, 128] for d in [0.2, 0.3] for lr in [1e-3, 5e-4]]

best_val_auprc, best_params, best_model = 0.0, None, None

for params in param_grid:
    print(f"\n--- hidden={params['hidden_size']}, dropout={params['dropout']}, lr={params['lr']} ---")
    set_all_seeds(RANDOM_SEED)
    model = SepsisLSTM(
        input_size=len(feature_cols),
        hidden_size=params['hidden_size'],
        num_layers=2,
        dropout=params['dropout'],
    )
    model, val_auprc = train_lstm(
        model, train_loader, val_loader,
        n_epochs=50, lr=params['lr'],
        pos_weight=pos_weight, patience=10, device=device,
    )
    print(f'Result: val AUPRC={val_auprc:.4f}')
    if val_auprc > best_val_auprc:
        best_val_auprc, best_params, best_model = val_auprc, params, model

print(f'Best: {best_params}  val AUPRC: {best_val_auprc:.4f}')

## 4. Evaluate and Log

In [ ]:
def get_probs_labels(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            logits  = model(X_batch.to(device), lengths)
            probs   = torch.sigmoid(logits)
            seq_len = logits.shape[1]
            mask = torch.zeros(y_batch.shape[0], seq_len, dtype=torch.bool)
            for i, l in enumerate(lengths):
                mask[i, :min(int(l), seq_len)] = True
            all_probs.extend(probs[mask].cpu().numpy())
            all_labels.extend(y_batch[:, :seq_len][mask].numpy())
    return np.array(all_probs), np.array(all_labels)

# Reuse loaders built in grid search cell
val_probs,  val_labels  = get_probs_labels(best_model, val_loader,  device)
test_probs, test_labels = get_probs_labels(best_model, test_loader, device)

threshold    = select_threshold(val_labels, val_probs)
val_metrics  = compute_all_metrics(val_labels,  val_probs,  threshold)
test_metrics = compute_all_metrics(test_labels, test_probs, threshold)

print(f'Test AUC-ROC: {test_metrics["auc_roc"]:.4f} | AUPRC: {test_metrics["auprc"]:.4f} | F1: {test_metrics["f1"]:.4f}')

log_results('C', 'LSTM', 'Strategy_A', val_metrics, test_metrics, best_params)

os.makedirs(MODELS_DIR, exist_ok=True)
torch.save(best_model.state_dict(), f'{MODELS_DIR}lstm_condition_C.pt')
joblib.dump(feature_cols, f'{MODELS_DIR}lstm_condition_C_feature_cols.pkl')
print(f'Model saved → {MODELS_DIR}lstm_condition_C.pt')